# TP : eQTL et Randomisation Mendélienne
## Magistère Européen de Génétique — M1 - Option Méthodologie en génétique humaine


<div style="background-color:#f8f9fa; padding:15px; border-left:4px solid #6c757d; margin-bottom:20px;">

**Remerciements**

Ce TP a été développé à partir des supports pédagogiques de **Marie Verbanck** (Institut Curie), que nous remercions chaleureusement pour son partage d'expertise en randomisation mendélienne et pour ses précieux conseils méthodologiques.

Les données GWAS proviennent du **GWAS Catalog** (EMBL-EBI) :
- LDL cholestérol : GCST90468080
- Maladie coronarienne : GCST005194

**Auteur** : Claire Vandiedonck  
**Annéede création** : 2025-2026  
**Formation** : M1 Magistère Européen de Génétique

</div>



## Objectifs pédagogiques


<div class="alert alert-block alert-info"> 

**Objectifs pédagogiques :**
1. Comprendre le concept d'eQTL et son rôle de variable instrumentale (IV)
2. Explorer des données de summary statistics GWAS réelles
3. Réaliser une analyse de Randomisation Mendélienne (MR) manuellement :
   - Calculer les estimateurs MR (IVW, MR-Egger, Weighted Median)
   - Visualiser et interpréter les résultats d'une analyse MR
   - Tester la robustesse des hypothèses par des analyses de sensibilité
    
**Durée : ~1h40** (après l'introduction magistrale)

</div>



## Contexte biologique et question de recherche
---

### LDL cholestérol et maladie coronarienne

Nous nous intéressons dans ce TP à la relation entre le **cholestérol LDL** (Low Density Lipoprotein cholesterol, LDL-C) et la **maladie coronarienne** (coronary artery disease, CAD).

<div class="alert alert-block alert-warning"> <b>Question centrale </b>: Le taux de LDL-C influence-t-il causalement le risque de développer une maladie coronarienne ?
</div>

Il est largement établi que des niveaux élevés de LDL-C constituent un facteur de risque majeur pour les maladies coronariennes. Plusieurs essais cliniques randomisés ont démontré de manière cohérente que la réduction du LDL-C, notamment avec les statines, diminue efficacement le risque de CAD.<br>


**Objectif du TP** : Utiliser la **randomisation mendélienne** pour estimer l'effet causal d'une diminution du LDL-C sur le risque de CAD, en s'affranchissant des biais des études observationnelles classiques. Ici  :
- l'exposition est le **cholestérol LDL**
- l'outcome est la **maladie coronarienne (CAD)**.


    
**Approche** :
- Partie I : Étudier un eQTL comme exemple d'instrument génétique (gène HMGCR)
- Parties II-IV : Analyse MR complète avec 280 instruments génétiques issus de GWAS

> ***Note sur les données*** : *Les données de ce TP sont synthétiques mais réalistes.  
> Les données eQTL sont simulées à partir des valeurs publiées dans GTEx v8 (tissu foie, gène *HMGCR*).  
> En revanche, les variables sinstruemenatles de la MR (IVs) sont basés sur de vrais rsIDs et des tailles d'effet tirées de la littérature (Willer et al. *Nat Genet* 2013 pour le LDL ; Nikpay et al. *Nat Genet* 2015 pour la CAD).*  
> *Le code pour reproduire les données depuis les sources originales est fourni sur le repository : https://github.com/CVandiedonck/MR-eQTL-env.*


### Principes de la randomisation mendélienne

La **randomisation mendélienne** (Mendelain Randomization (MR))repose sur l'idée que les variants génétiques sont aléatoirement distribués dans la population (comme dans un essai randomisé). Pour qu'un SNP soit un bon **instrument génétique**, il doit satisfaire trois hypothèses :

1. **Relevance** : le SNP est associé à l'exposition (ici : il affecte l'expression d'un gène)
2. **Indépendance** : le SNP est indépendant des facteurs confondants
3. **Exclusion restriction** : le SNP n'affecte l'outcome que *via* l'exposition

Un **eQTL** (*expression Quantitative Trait Locus*) est exactement ça : un SNP dont on sait qu'il est associé à la régulation de l'expression d'un gène. Il satisfait naturellement l'hypothèse 1.

## 0. Setup (~5 min)
---

### A. Chargement de `rpy2` et installation des packages R si nécessaire.

In [ ]:
## cell 1
%load_ext rpy2.ipython

In [ ]:
%%R
## cell 2

# Installation des packages si nécessaire (à ne lancer qu'une fois)
if (!require("ggplot2", quietly=TRUE)) {
    install.packages("ggplot2", repos="https://cloud.r-project.org")
}
if (!require("dplyr", quietly=TRUE)) {
    install.packages("dplyr",   repos="https://cloud.r-project.org")
}

library(dplyr)
library(ggplot2)
print(sessionInfo())

### B. Récupérer les données pour ce TP

Travaillez dans un répertoire de travail `TP_MR`

Copiez les données dans un sous-répertoire `data` que vous créez.

In [ ]:
%%bash
## cell 3

# Créez le dossier data/ à côté de ce notebook
mkdir -p data

# Copiez les fichiers depuis le partage
cp /srv/data/Datasets/MendelianRandomization/*.csv data/

Vérifiez que vous avez bien les 2 fichiers :

In [ ]:
%%bash
## cell 4

ls -lh data/

Vous devriez voir :
- `eqtl_HMGCR.csv` (~30 KB)
- `MR_IVs_LDL_CAD.csv` (~47 KB)

**Note** : Tous les graphiques et résultats s'affichent directement dans le notebook, pas besoin de créer d'autres dossiers.


## Partie I — eQTL comme variable instrumentale (~30 min)
---


### A. Notre exemple : *HMGCR* et le cholestérol LDL

Le gène *HMGCR* code la HMG-CoA réductase, enzyme clé de la voie de biosynthèse du cholestérol. Cette enzyme est la cible pharmacologique des statines : en inhibant *HMGCR*, les statines diminuent la production de cholestérol et abaissent ainsi le taux de LDL sanguin.

Le SNP **rs12916** (chr5:74,656,528, build GRCh37, allèles T>C, MAF=0.4) est associé à l'expression de *HMGCR* (https://www.gtexportal.org/home/gene/ENSG00000113161.17/eQTLsTab) et au taux de LDL cholestérol (https://www.ebi.ac.uk/gwas/variants/rs12916). L'allèle T (majeur) diminue à la fois l'expression de HMGCR et le LDL circulant, mimant l'effet des statines : moins d'enzyme → moins de cholestérol produit → LDL plus bas.

Ce variant génétique constitue donc un **instrument génétique** idéal pour estimer l'effet causal d'une diminution du LDL sur le risque cardiovasculaire.

Nous allons charger les données d'eQTL `eqtl_HMGCR.csv` de ce SNP. 

In [ ]:
%%R
## cell 5

# Chargement des données eQTL
eqtl <- read.csv("data/eqtl_HMGCR.csv")
str(eqtl)
head(eqtl)

Le tableau `eqtl` contient **250 individus** avec les colonnes suivantes :

##### Variables principales
- **`individual_id`** : Identifiant unique (1 à 250)
- **`genotype_rs12916`** : Nombre d'allèles C (mineur) pour le SNP rs12916 (0, 1 ou 2 copies). Note : L'allèle T est protecteur (diminue HMGCR et LDL), C est à risque (augmente HMGCR et LDL)
- **`HMGCR_expression`** : Expression de HMGCR normalisée (log2-transformed)

##### Covariables
- **`age`** : Âge en années (20-70)
- **`sex`** : Sexe (0 = femme, 1 = homme)
- **`PC1`** : Premier axe de l'analyse en composantes principales (ACP) sur les génotypes
- **`PC2`** : Deuxième axe de l'ACP
- **`BMI`** : Indice de masse corporelle (kg/m²)
- **`LDL_cholesterol`** : Taux de cholestérol LDL (mmol/L)
- **`CAD_status`** : Présence de maladie coronarienne (0 = non, 1 = oui)

<div style="background-color:#e8f4f8; padding:10px; border-radius:4px;">

**Question 1a** — Décrivez la distribution des génotypes pour rs12916.  
Combien d'individus ont 0, 1 ou 2 copies de l'allèle C ?  
Comparez à la fréquence allélique attendue sous Hardy-Weinberg (MAF = 0.4).

</div>

<details>
<summary><b>💡 Indice</b></summary>

Pour tester Hardy-Weinberg :
1. Calculer les effectifs observés avec `table()`
2. Calculer les proportions attendues : p²(CC), 2pq(CT), q²(TT) (et les effectifs si vous voulez)
3. Test chi² : `chisq.test(obs, p = proportions_attendues)`

</details>

In [ ]:
%%R
## cell 6
# Votre réponse ici


<div style="background-color:#e8f4f8; padding:10px; border-radius:4px;">

**Question 1b** — Visualisez la relation entre le génotype rs12916 et l'expression de *HMGCR*.  
Quel est le sens de l'effet ? Est-ce cohérent avec la biologie ?

</div>

<details>
<summary><b>💡 Indice 1 : Type de graphique</b></summary>

Utilisez un **boxplot** pour comparer l'expression selon les 3 génotypes (TT, TC, CC).

En R base : `boxplot(expression ~ génotype, data = ...)`  
En ggplot2 : `geom_boxplot()` ou `geom_violin()`

</details>

<details>
<summary><b>💡 Indice 2 : Préparer les données</b></summary>

Créez un **facteur** avec des labels explicites :
```r
eqtl$genotype_factor <- factor(eqtl$genotype_rs12916,
                                levels = 0:2,
                                labels = c("TT", "TC", "CC"))
```

</details>

<details>
<summary><b>💡 Indice 3 : Interprétation biologique</b></summary>

<b>Attendu</b> : L'allèle T devrait **diminuer** l'expression de HMGCR.

<b>Pourquoi ?</b> HMGCR est la cible des statines. Une diminution de HMGCR réduit la production de cholestérol, ce qui abaisse le LDL (effet bénéfique similaire aux statines).

Observez la tendance visuelle sur le graphique : l'expression semble-t-elle baisser avec le nombre d'allèles T ?

</details>

In [ ]:
%%R
## cell 7
# Votre réponse ici


### B. Test statistique de l'association génotype-expression

Le graphique suggère une diminution de l'expression avec le nombre d'allèles T. 
Testons si cette tendance est statistiquement significative.

<div style="background-color:#e8f4f8; padding:10px; border-radius:4px;">

**Question 1c** — Testez statistiquement l'association eQTL par régression linéaire.  
Ajustez sur l'âge, le sexe et PC1.  
Quelle est la taille d'effet (beta) ? Est-ce significatif ?

</div>

<details>
<summary><b>💡 Indice 1 : Syntaxe de la régression</b></summary>

Utilisez `lm()` pour une régression linéaire multiple :
```r
model <- lm(variable_réponse ~ variable_explicative + covariable1 + covariable2, 
            data = tableau)
summary(model)
```

</details>

<details>
<summary><b>💡 Indice 2 : Quelles covariables choisir ?</b></summary>

**Réfléchissez** : Quelles variables du tableau `eqtl` devraient être incluses comme covariables ?

**Critères de sélection** :
- Variables qui pourraient **confondre** l'association génotype-expression
- Variables liées à la **structure de population** ou aux **caractéristiques démographiques**
- ⚠️ **Éviter** les variables qui sont des **conséquences** du génotype (médiateurs ou outcomes)

Consultez la description des données pour vous aider.

</details>

<details>
<summary><b>💡 Indice 3 : Interpréter les résultats</b></summary>

Dans la sortie de `summary()`, regardez la ligne `genotype_rs12916` :

- **`Estimate`** : taille d'effet (beta) = changement d'expression par allèle T
- **`Pr(>|t|)`** : p-value de l'association
- Un beta **négatif** indique que T **diminue** l'expression

**Seuil de significativité** : p < 5e-8

</details>

In [ ]:
%%R
## cell 8
# Votre réponse ici


---

<div style="background-color:#d1ecf1; padding:12px; border-radius:4px; border-left:4px solid #0c5460;">

<h3 style="color:#0c5460; margin-top:0; font-weight;">📚 Pour aller plus loin : Choix des covariables en analyse eQTL</h3>

<br>
    
Dans la régression ci-dessus, nous avons ajusté sur `age`, `sex`, `PC1` et `PC2`. 
Mais le tableau contient d'autres variables (`BMI`, `LDL_cholesterol`, `CAD_status`). 
<br>
    <b>Pourquoi ne pas les inclure ? </b>

<br>
  

<div style="margin-left:15px;">

#### ✅ Variables ajustées : `age`, `sex`, `PC1`, `PC2`

</div>

Raison : Ce sont des confondants potentiels indépendants du génotype qui peuvent affecter l'expression de *HMGCR*. On les inclut.

<br>

<div style="margin-left:15px;">

#### ❌ Variables non incluses dans l'ajustement : `BMI`, `LDL_cholesterol`, `CAD_status`

</div>

**1. `LDL_cholesterol` — Variable MÉDIATRICE**

Chaîne causale : `rs12916 → HMGCR expression ↓ → LDL production ↓`

- Le génotype affecte le LDL via l'expression de HMGCR
- Ajuster sur LDL casserait la chaîne causale qu'on étudie

**2. `CAD_status` — Variable OUTCOME (collider)**

Chaîne causale : `rs12916 → HMGCR → LDL ↓ → Risque CAD ↓`

- CAD est une conséquence du génotype (via LDL)
- Ajuster sur CAD créerait un biais de sélection

**3. `BMI` — Acceptable mais non nécessaire**

- Indépendant du génotype (randomisation mendélienne)

---

<div style="background-color:#e8f4f8; padding:10px; border-radius:4px;">

<h4 style="color:#0277bd; margin-top:0;">🎓 Question bonus</h4>

Que se passe-t-il si on ajuste sur `LDL_cholesterol` ? Comparez l'effet du génotype avec et sans ajustement sur LDL.

</div>

</div>

---

In [ ]:
%%R
## cell 9
# Votre réponse ici

<div style="background-color:#fff3cd; padding:12px; border-left:4px solid #ffc107; border-radius:4px;">

<h3 style="color:#f57c00; margin-top:0;">📖 Règle générale en analyse eQTL/GWAS</h3>

**Ajuster sur** :
- Facteurs techniques (batch, plateforme de séquençage)
- Structure de population (PC1, PC2, PC3...)
- Covariables populationnelles non-médiées (âge, sexe)

**Ne PAS ajuster sur** :
- **Médiateurs** : variables sur le chemin causal entre génotype et outcome
- **Outcomes/conséquences** : colliders qui créent des biais de sélection
- **Variables post-exposition** : mesurées après l'effet du génotype

</div>

<div style="background-color:#e8f4f8; padding:10px; border-radius:4px;">

**Question 1d** — Calculez la **statistique F** de l'instrument :
$$F = \frac{\hat{\beta}^2}{SE^2}$$
Une règle empirique : F > 10 indique un instrument fort. Qu'en est-il ici ?  
Interprétez : qu'est-ce qu'un instrument *faible* biaiserait dans l'analyse MR ?

</div>

<details>
<summary><b>💡 Indice 1 : Extraire β et SE du modèle</b></summary>

Utilisez le modèle ajusté `mod_adj` :
```r
# Extraire les coefficients
coefs <- summary(mod_adj)$coefficients

# Pour genotype_rs12916 :
beta <- coefs["genotype_rs12916", "Estimate"]
se <- coefs["genotype_rs12916", "Std. Error"]
```

</details>

<details>
<summary><b>💡 Indice 2 : Interpréter le seuil F > 10</b></summary>

**Règle de Staiger & Stock (1997)** :
- **F < 10** : Instrument faible (weak instrument)
- **F > 10** : Instrument fort (strong instrument)

Plus F est élevé, plus l'association génotype-exposition est forte.

**Attendu ici** : Avec p < 5e-8, on devrait avoir F >> 10 !

</details>

<details>
<summary><b>💡 Indice 3 : Conséquences d'un instrument faible</b></summary>

**Un instrument faible (F < 10) entraîne** :

1. **Biais vers la nullité** : Sous-estimation de l'effet causal (perte de puissance)
2. **Sensibilité aux facteurs confondants** : L'analyse MR perd son avantage (élimination des biais) et redevient sensible aux confusions résiduelles comme une étude observationnelle classique
3. **Estimations imprécises** : Intervalles de confiance très larges
4. **Violation des hypothèses MR** : L'instrument est trop faible pour isoler l'effet causal

**En pratique** : Si F < 10, l'analyse MR n'est pas fiable.

</details>

In [ ]:
%%R
## cell 10
# Votre réponse ici

---
## Partie II — Exploration des données MR (~30 min)

### A. Passage à l'échelle : du gène au GWAS

L'approche eQTL sur un gène est illustrative, mais une MR robuste utilise **de nombreux instruments génétiques** tirés d'un GWAS sur l'exposition.

Le fichier `MR_IVs_LDL_CAD.csv` contient 280 SNPs instrumentaux déjà sélectionnés et harmonisés :
- Associés au LDL à *p* < 5×10⁻⁸ (seuil GWAS)
- Indépendants (clumpés, r² < 0.01, fenêtre 10 Mb)
- Effets alignés sur l'allèle *augmentant* le LDL
- Présents dans les deux GWAS (LDL et CAD)


<div style="background-color:#d1ecf1; padding:12px; border-radius:4px; border-left:4px solid #0c5460;">

<h3 style="color:#0c5460; margin-top:0; font-weight:">📊 Pour info : Génération du fichier MR_IVs_LDL_CAD.csv</h3>

Ce fichier contient **280 SNPs indépendants** servant d'instruments génétiques pour étudier l'effet causal du LDL cholestérol sur la maladie coronarienne (CAD). Voici les grandes étapes de sa création :

#### **1. Sources GWAS**

- Exposition (LDL cholestérol) :
    - GWAS Catalog ID : GCST90468080
    - 7,9 millions de SNPs, build GRCh37

- Outcome (Maladie coronarienne) :
    - GWAS Catalog ID : GCST005194  
    - 13,3 millions de SNPs, build GRCh37

#### **2. Pipeline de génération (outil `genal-python`)**

**Étapes automatisées avec genal** :

1. **Chargement** : Import des summary statistics dans des objets `genal.Geno()`

2. **Preprocessing** : Nettoyage des données (valeurs manquantes, format, cohérence allélique) avec `preprocess_data()`

3. **Clumping** : Sélection des instruments = SNPs indépendants pour l'exposition avec `E_Geno.clump()`
   - Seuil de significativité : p < 5×10⁻⁸ (genome-wide)
   - Indépendance : r² < 0.01 (LD faible)
   - Fenêtre : 10 Mb
   - Panel de référence : 1000 Genomes EUR (build 37), téléchargé automatiquement

4. **Harmonisation** : Alignement des allèles effet entre exposition et outcome
   - Gestion des SNPs palindromiques (action=2, conservatif)
   - Flip des allèles si nécessaire

5. **Annotation des gènes** : Identification des gènes proches (±500kb) avec GENCODE v19
   - Priorisation des gènes *protein-coding*
   - 280/280 SNPs annotés
    
6. **Calcul de la F-statistic** : Mesure de la force de chaque instrument
   - Formule : F = β²_LDL / SE²_LDL
   - Évalue l'association SNP → LDL (exposition)
   - Seuil F > 10 = instrument fort (Staiger & Stock, 1997)
   - **Note** : Cette statistique est calculée à partir des données GWAS (β et SE), elle n'est pas directement fournie par les summary statistics. On ne la calcule que pour l'instrument et pas pour l'outcome.
    
    
#### **3. Résultat final**

- **Fichier** : `data/MR_IVs_LDL_CAD.csv`  
- **Contenu** : 280 instruments génétiques avec :
    - Effets sur LDL (β, SE, p-value)
    - **F-statistic** : calculée comme F = β²/SE² (mesure la force de l'instrument)
    - Effets sur CAD (β, SE, p-value)
    - Annotation génique (colonne GENE)
    
**Gènes clés identifiés** : *HMGCR*, *PCSK9*, *LDLR*, *APOE*, *APOB*, *CETP*, *LPA*, *LIPC*, *LIPG*

---

**Outils utilisés** :
- **genal-python** ([GitHub](https://github.com/legenepi/genal)) : Preprocessing, clumping, harmonisation
- **GenomicRanges** (Bioconductor) : Annotation génomique
- **GENCODE v19** : Annotations gènes (build GRCh37)

**Pour reproduire** : Voir `README_DATA_GENERATION.md` dans le dépôt GitHub

</div>

---

Chargeons maintenant ces données pour explorer les instruments génétiques :

In [ ]:
%%R
# cell 11

# Chargement des données MR
mr_data <- read.csv("data/MR_IVs_LDL_CAD.csv")
str(mr_data)
head(mr_data[, c("SNP","GENE","BETA_LDL","SE_LDL","P_LDL","BETA_CAD","SE_CAD","F_stat")])

<div style="background-color:#fff3cd; padding:10px; border-left:4px solid #ffc107; border-radius:4px;">

**📌 Note sur la F-statistic**

La colonne `F_stat` dans ce fichier mesure la **force de chaque instrument génétique**. Elle a été calculée à partir des données GWAS sur le LDL :

$$F = \frac{\beta^2_{LDL}}{SE^2_{LDL}}$$

**Points importants** :
- F mesure uniquement l'association SNP → LDL (exposition), **pas** SNP → CAD
- Seuil : F > 10 indique un instrument fort (Staiger & Stock, 1997)
- Cette statistique n'est pas fournie directement par les summary statistics GWAS, elle a été calculée lors de la génération du fichier   

</div>

<div style="background-color:#e8f4f8; padding:10px; border-radius:4px;">

**Question 2a** — Quelle est la distribution des statistiques F dans ce jeu de données ?  
Tous les instruments sont-ils forts ? Notez les gènes connus (*HMGCR*, *PCSK9*, *LDLR*, *APOE*) : sont-ils présents ?


<details>
<summary><b>💡 Indice 1 : Visualiser la distribution des F</b></summary>

Utilisez un histogramme pour visualiser la distribution des F-statistics :
```r
hist(mr_data$F_stat, 
     breaks=30,
     main="Distribution des F-statistics",
     xlab="F-statistic",
     ylab="Nombre de SNPs")
abline(v=10, col="red", lwd=2, lty=2)  # Seuil F > 10
```
<br>
Ou calculez des statistiques descriptives :
    
```r
summary(mr_data$F_stat)
```

</details>

<details>
<summary><b>💡 Indice 2 : Compter les instruments faibles</b></summary>

Pour savoir combien de SNPs ont F < 10 (instruments faibles) :
```r
# Classification et comptage
table(ifelse(mr_data$F_stat >= 10, "Fort (F≥10)", "Faible (F<10)"))

# Proportions en %
round(100 * prop.table(table(mr_data$F_stat >= 10)), 1)
```

</details>

<details>
<summary><b>💡 Indice 3 : Rechercher les gènes connus</b></summary>

Pour vérifier la présence de gènes spécifiques :
```r
# Gènes à chercher
genes_cles <- c("HMGCR", "PCSK9", "LDLR", "APOE")

# Chercher dans la colonne GENE
# Compter et calculer F moyen par gène
comptage <- aggregate(F_stat ~ GENE, data = genes_cles, FUN = length)
f_moyen <- aggregate(F_stat ~ GENE, data = genes_cles, FUN = mean)

# Fusionner
resultats <- merge(comptage, f_moyen, by = "GENE", suffixes = c("_count", "_mean"))
names(resultats) <- c("GENE", "Nb_SNPs", "F_moyen")
print(resultats)
```

</details>

In [ ]:
%%R
# cell 12
# Votre réponse ici


### B. Représentation classique MR multi-instrument

Dans les études de randomisation mendélienne avec plusieurs instruments, il est conventionnel de représenter graphiquement tous les SNPs comme "augmentant l'exposition" pour faciliter la lecture. Pour ce faire, on transforme les axes pour que l'axe X (effet sur LDL) soit toujours positif.

**Convention** : Les analyses statistiques utilisent les valeurs originales (signées). Cette transformation concerne uniquement la visualisation.

---

<div style="background-color:#e8f4f8; padding:10px; border-radius:4px;">

**Question 2b** — Faites un scatter plot des effets SNP sur CAD en fonction des effets SNP sur LDL (après transformation en valeur absolue).
Que remarquez-vous ? Quelle relation attendez-vous si LDL cause la CAD ?
</div>

    
<details>
<summary><b>💡 Indice 1 : Créer le scatter plot</b></summary>
    
**Transformation MR** : Tous les SNPs codés pour augmenter le LDL
```r
# X = valeur absolue de BETA_LDL (tous positifs)
# Y = BETA_CAD ajusté selon le signe de BETA_LDL
x <- abs(mr_data$BETA_LDL)
y <- mr_data$BETA_CAD * sign(mr_data$BETA_LDL)
```    

    
**Version R base (simple)** :
```r
plot(abs(mr_data$BETA_LDL), 
     mr_data$BETA_CAD * sign(mr_data$BETA_LDL),
     xlab = "Effet sur LDL (valeur absolue)",
     ylab = "Effet sur CAD",
     main = "Relation SNP-LDL vs SNP-CAD",
     pch = 16, col = rgb(0, 0, 0, 0.5))
abline(h=0, col="gray", lty=2)
```

**Version ggplot2 (avancée)** :
```r
p <- ggplot(mr_data, aes(x = abs(BETA_LDL), 
                          y = BETA_CAD * sign(BETA_LDL))) +
    geom_point(alpha=0.6) +
    geom_hline(yintercept=0, linetype="dotted") +
    labs(x="Effet sur LDL", y="Effet sur CAD") +
    theme_minimal()

print(p)  # Important : print() explicite dans %%R !
```
</details>

<details>
<summary><b>💡 Indice 2 : Relation causale attendue</b></summary>

Formule théorique : β_CAD = θ × β_LDL, où θ est l'effet causal du LDL sur CAD    
    
Si LDL cause la CAD, on s'attend à :

- Relation linéaire positive : Les SNPs qui augmentent le LDL (β_LDL > 0) devraient augmenter le risque de CAD (β_CAD > 0)
- Pente proportionnelle : Plus l'effet sur LDL est fort, plus l'effet sur CAD devrait être important
- Origine (0,0) : Les SNPs sans effet sur LDL n'ont pas d'effet sur CAD 
    
</details>

<details>
<summary><b>💡 Indice 3 : Ajouter une droite de tendance</b></summary>

Pour visualiser la tendance générale, utilisez `I()` pour forcer l'évaluation dans la formule :
```r
fit <- lm(I(BETA_CAD * sign(BETA_LDL)) ~ 0 + I(abs(BETA_LDL)), data = mr_data)
abline(fit, col="red", lwd=2)
```

Cette pente estime l'effet causal moyen du LDL sur la CAD.

</details>

In [ ]:
%%R
# cell 13
# Votre réponse ici

---
## Partie III — Calcul de l'effet causal (IVW) (~30 min)

Nous avons identifié 280 instruments génétiques et vérifié leur force (F>10). Nous allons à présent estimer l'effet causal du LDL sur la CAD en utilisant la méthode ***Inverse Variance Weighted (IVW)***, qui combine l'information de tous les instruments en pondérant par leur précision.

### Principe : du Wald ratio à l'IVW

Pour chaque SNP *j*, le **ratio de Wald** estime l'effet causal de l'exposition sur l'outcome :

$$\hat{\theta}_j = \frac{\hat{\beta}_{Y,j}}{\hat{\beta}_{X,j}}$$

Son erreur standard approximée est :

$$SE(\hat{\theta}_j) \approx \frac{SE_{Y,j}}{|\hat{\beta}_{X,j}|}$$

La méthode **Inverse Variance Weighted (IVW)** combine ces estimés en une moyenne pondérée par l'inverse de la variance :

$$\hat{\theta}_{IVW} = \frac{\sum_j w_j \hat{\theta}_j}{\sum_j w_j} \quad \text{avec} \quad w_j = \frac{1}{SE(\hat{\theta}_j)^2}$$




<div style="background-color:#d1ecf1; padding:12px; border-radius:4px; border-left:4px solid #0c5460;">

<h3 style="color:#0c5460; margin-top:0; font-weight;">📊 Échelles d'interprétation et avantages de l'IVW</h3>

### Échelles selon le type d'outcome

L'**interprétation du ratio de Wald** (et de l'IVW) dépend de la nature de l'outcome :

**Outcome continu** (ex: pression artérielle, IMC, glycémie)
- β_outcome = **unités linéaires** (mmHg, kg/m², mmol/L)
- θ = **changement absolu** de l'outcome par SD d'exposition
- Exemple : "1 SD ↑ exposition → +2.5 mmHg de pression systolique"

**Outcome binaire** (ex: maladie coronarienne, diabète, décès)
- β_outcome = **log(OR)** (régression logistique dans le GWAS)
- θ = **log(OR)** de l'outcome par SD d'exposition
- Conversion : OR = exp(θ)
- **Notre cas** : CAD (binaire) → θ en log(OR) ✓

---

### Pourquoi l'IVW est meilleur que les ratios individuels ?

**Chaque SNP donne une estimation θ_j**, mais :

❌ **Problèmes des ratios individuels** :
- Très **imprécis** (erreurs standard élevées)
- Affectés par la **pleïotropie** de chaque SNP spécifique
- Difficile de choisir **quel SNP croire**
- Variabilité importante (comme vu dans le forest plot)

✅ **Avantages de l'IVW** :
- **Combine tous les instruments** → utilise toute l'information disponible
- **Pondère par la précision** (poids = 1/SE²) → les SNPs précis comptent plus
- **Réduit l'erreur aléatoire** → estimation plus stable et précise
- **Robuste** si la pleïotropie est équilibrée entre instruments
- **Augmente la puissance statistique** → p-values plus fiables


L'IVW est l'**estimateur standard** en MR avec plusieurs instruments.

</div>

---

<div style="background-color:#e8f4f8; padding:10px; border-radius:4px;">

**Question 3a** — Calculez le ratio de Wald et son SE pour chaque SNP.  
Quel SNP a le ratio le plus élevé ? le plus faible ?

</div>

<details>
<summary><b>💡 Indice 1 : Formule du ratio de Wald</b></summary>

Le ratio de Wald pour chaque SNP estime l'effet causal individuel :

$$
\theta_j = \frac{\beta_{CAD,j}}{\beta_{LDL,j}}
$$

**En R** :
```r
ratio_wald <- mr_data$BETA_CAD / mr_data$BETA_LDL
```

**Interprétation** : Changement du log(OR) de CAD par unité d'augmentation du LDL pour ce SNP.

</details>

<details>
<summary><b>💡 Indice 2 : Erreur standard du ratio de Wald</b></summary>

**Approximation du premier ordre** (méthode delta) :

$$
SE(\theta_j) \approx \frac{SE_{CAD,j}}{|\beta_{LDL,j}|}
$$

**En R** :
```r
se_wald <- mr_data$SE_CAD / abs(mr_data$BETA_LDL)
```

**Note** : Cette formule suppose que l'incertitude sur β_LDL est négligeable devant celle sur β_CAD (généralement vrai car échantillons LDL plus grands).

</details>

<details>
<summary><b>💡 Indice 3 : Identifier les SNPs extrêmes</b></summary>

Pour trouver les SNPs avec les ratios extrêmes, utilisez `order()` :
```r
# Top 3 ratios les plus élevés
top_idx <- order(ratio_wald, decreasing=TRUE)[1:3]
mr_data[top_idx, c("SNP", "GENE", "ratio_wald")]

# Top 3 ratios les plus faibles
bottom_idx <- order(ratio_wald)[1:3]
mr_data[bottom_idx, c("SNP", "GENE", "ratio_wald")]
```

</details>

In [ ]:
%%R
## cell 14
# Votre réponse ici


<div style="background-color:#e8f4f8; padding:10px; border-radius:4px;">

**Question 3b** — Construisez un **forest plot** des ratios de Wald.  
Que révèle l'hétérogénéité entre les instruments ?

</div>

<details>
<summary><b>💡 Indice 1 : Calculer les intervalles de confiance</b></summary>

Pour chaque SNP, calculez l'IC à 95% :

$$
IC_{95\%} = \theta_j \pm 1.96 \times SE(\theta_j)
$$

**En R** :
```r
mr_data$wald_lower <- mr_data$ratio_wald - 1.96 * mr_data$se_wald
mr_data$wald_upper <- mr_data$ratio_wald + 1.96 * mr_data$se_wald
```

</details>

<details>
<summary><b>💡 Indice 2 : Créer le forest plot</b></summary>

**Version R base** : Utiliser `segments()` pour les IC et `points()` pour les estimés
```r
# Ordonner par ratio
ord <- order(mr_data$ratio_wald)
n <- nrow(mr_data)

plot(mr_data$ratio_wald[ord], 1:n, 
     xlim=range(c(mr_data$wald_lower, mr_data$wald_upper), na.rm=TRUE),
     xlab="Ratio de Wald", ylab="SNP (index)", pch=16)
segments(mr_data$wald_lower[ord], 1:n, 
         mr_data$wald_upper[ord], 1:n)
abline(v=median(mr_data$ratio_wald), col="red", lty=2)
```

**Version ggplot2** : Utiliser `geom_pointrange()`

</details>

<details>
<summary><b>💡 Indice 3 : Interpréter l'hétérogénéité</b></summary>

**Hétérogénéité élevée** si :
- Les IC ne se chevauchent pas tous
- Certains SNPs ont des ratios très différents de la médiane
- Certains IC ne contiennent pas la médiane

**Causes possibles** :
- Pleïotropie horizontale (SNPs affectent CAD par d'autres voies)
- Erreurs de mesure
- Différences biologiques entre instruments

L'hétérogénéité sera testée formellement avec la statistique de Cochran Q.

</details>

In [ ]:
%%R
## cell 15
# Votre réponse ici


<div style="background-color:#e8f4f8; padding:10px; border-radius:4px;">

**Question 3c** — Calculez l'estimé IVW manuellement (moyenne pondérée des ratios de Wald).  
Calculez le SE, l'IC 95% et la p-value.  
Interprétez : est-ce que le LDL cause la CAD ?

</div>

<details>
<summary><b>💡 Indice 1 : Formule IVW</b></summary>

L'estimateur **Inverse Variance Weighted** est une moyenne pondérée des ratios de Wald :

$$
\theta_{IVW} = \frac{\sum w_j \theta_j}{\sum w_j}
$$

où les poids sont :

$$
w_j = \frac{1}{SE(\theta_j)^2}
$$

**En R** :
```r
weights <- 1 / mr_data$se_wald^2
theta_ivw <- sum(weights * mr_data$ratio_wald) / sum(weights)
```

</details>

<details>
<summary><b>💡 Indice 2 : Erreur standard de l'IVW</b></summary>

L'erreur standard de l'IVW est :

$$
SE(\theta_{IVW}) = \frac{1}{\sqrt{\sum w_j}}
$$

**En R** :
```r
se_ivw <- 1 / sqrt(sum(weights))
```

</details>

<details>
<summary><b>💡 Indice 3 : IC et p-value</b></summary>

**Intervalle de confiance à 95%** :

$$
IC_{95\%} = \theta_{IVW} \pm 1.96 \times SE(\theta_{IVW})
$$

**P-value** (test de Wald, H₀ : θ = 0) :

$$
z = \frac{\theta_{IVW}}{SE(\theta_{IVW})}
$$
```r
# IC 95%
ic_lower <- theta_ivw - 1.96 * se_ivw
ic_upper <- theta_ivw + 1.96 * se_ivw

# P-value (test bilatéral)
z_stat <- theta_ivw / se_ivw
p_value <- 2 * (1 - pnorm(abs(z_stat)))
```

</details>

In [ ]:
%%R
## cell 16
# Votre réponse ici


<div style="background-color:#e8f4f8; padding:10px; border-radius:4px;">

**Question 3d** — Vérifiez votre résultat par régression WLS sans intercept.  
Les deux méthodes donnent-elles exactement le même résultat ? Pourquoi ?

</div>

<details>
<summary><b>💡 Indice 1 : Régression WLS (Weighted Least Squares)</b></summary>

L'IVW peut être calculé comme une régression linéaire pondérée de β_CAD sur β_LDL sans intercept :

$$
\beta_{CAD} = \theta \times \beta_{LDL} + \epsilon
$$

**En R** :
```r
wls_model <- lm(BETA_CAD ~ 0 + BETA_LDL, 
                weights = 1/SE_CAD^2, 
                data = mr_data)
```

Le **0** dans la formule force la régression à passer par l'origine (pas d'intercept).

</details>

<details>
<summary><b>💡 Indice 2 : Extraire les résultats</b></summary>

Pour comparer avec l'IVW manuel :
```r
# Coefficient (pente)
theta_wls <- coef(wls_model)["BETA_LDL"]

# Erreur standard
se_wls <- summary(wls_model)$coefficients["BETA_LDL", "Std. Error"]

# P-value
p_wls <- summary(wls_model)$coefficients["BETA_LDL", "Pr(>|t|)"]

# IC 95%
confint(wls_model)
```

</details>


In [ ]:
%%R
## cell 17
# Votre réponse ici
  


<summary><b> <mark>Pourquoi les résultats sont-ils similaires ?</b></summary></mark>

L'estimé θ est identique car les deux méthodes font la même chose :

- **IVW manuel** : Moyenne pondérée des ratios de Wald
- **Régression WLS** : Meilleure droite passant par l'origine

Les SE peuvent différer légèrement selon les poids utilisés dans la régression, mais **l'effet causal estimé est le même**. La régression est plus pratique car R calcule automatiquement SE, IC et p-value.

</details>

---

## Partie IV — Pour aller plus loin : Analyses de sensibilité

<div style="background-color:#d1ecf1; padding:12px; border-radius:4px; border-left:4px solid #0c5460;">

**📚 Section bonus** : Cette partie présente des analyses de sensibilité avancées pour tester la robustesse de l'estimation IVW. Le code complet et les interprétations sont fournis. Vous pouvez exécuter ces analyses pour approfondir votre compréhension de la MR.

</div>



### A. Qu'est-ce qu'une analyse de sensibilité ?

Une **analyse de sensibilité** teste la **robustesse** d'un résultat statistique en vérifiant s'il reste valide lorsque certaines hypothèses sont violées ou assouplies.

**En MR, les analyses de sensibilité répondent à la question** :  
*"Notre conclusion (LDL → CAD) reste-t-elle vraie si certains instruments violent les hypothèses de la MR ?"*

**Principe général** :
1. Calculer l'estimé principal (ici : IVW) sous **hypothèses strictes**
2. Recalculer avec des **méthodes alternatives** qui tolèrent certaines violations
3. **Comparer** : Si les résultats convergent → conclusion robuste ✓

**Analogie** : Tester si une voiture fonctionne non seulement sur route parfaite, mais aussi sur gravier, en montée, avec du vent... Si elle performe dans tous les cas, c'est une voiture robuste !



### B. Pourquoi en MR spécifiquement ?

L'estimateur IVW suppose que **tous les instruments sont valides**, c'est-à-dire qu'ils satisfont les trois hypothèses de la MR :
1. **Relevance** : Associés à l'exposition (✓ testé avec F > 10)
2. **Exogénéité** : Indépendants des facteurs confondants
3. **Exclusion restriction** : N'affectent l'outcome QUE via l'exposition

En pratique, **certains SNPs peuvent violer l'hypothèse 3** (pleïotropie horizontale : ils affectent la CAD par d'autres voies que le LDL). Les analyses de sensibilité testent si notre conclusion reste valide malgré ces violations potentielles.

### C. Trois approches complémentaires

| Méthode | Robustesse | Hypothèse |
|---------|-----------|-----------|
| **Test Q** | Détecte l'hétérogénéité | - |
| **MR-Egger** | Pleïotropie directionnelle | Pleïotropie indépendante de β_X |
| **Weighted Median** | Pleïotropie non-directionnelle | >50% instruments valides |


#### 1. Test d'hétérogénéité (Cochran Q)

Le **test de Cochran Q** évalue si les ratios de Wald individuels sont **homogènes** ou s'il existe une **hétérogénéité** significative entre instruments. Une hétérogénéité élevée suggère que certains SNPs ont des effets causaux différents, possiblement à cause de pleïotropie.

**Formule** :

$$
Q = \sum_{j=1}^{k} w_j (\theta_j - \theta_{IVW})^2
$$

où :
- $\theta_j$ = ratio de Wald du SNP $j$
- $\theta_{IVW}$ = estimé IVW (moyenne pondérée)
- $w_j = 1/SE(\theta_j)^2$ = poids

**Sous H₀** (homogénéité) : $Q \sim \chi^2(k-1)$

**Statistique I²** : Pourcentage de la variance totale due à l'hétérogénéité réelle (vs hasard).

$$
I^2 = \frac{Q - (k-1)}{Q} \times 100\%
$$

- I² < 25% : hétérogénéité faible
- I² = 25-75% : hétérogénéité modérée
- I² > 75% : hétérogénéité élevée



In [ ]:
%%R
## cell 18

# Calcul de la statistique Q
weights <- 1 / mr_data$se_wald^2
theta_ivw_val <- sum(weights * mr_data$ratio_wald) / sum(weights)

# Q = somme des écarts pondérés au carré
Q <- sum(weights * (mr_data$ratio_wald - theta_ivw_val)^2)

# Degrés de liberté
df_q <- nrow(mr_data) - 1

# P-value
p_q <- pchisq(Q, df=df_q, lower.tail=FALSE)

# I² (% de variance due à l'hétérogénéité plutôt qu'au hasard)
I2 <- max(0, (Q - df_q) / Q) * 100

cat("Statistique Q :", round(Q, 2), "\n")
cat("Degrés de liberté :", df_q, "\n")
cat("P-value :", format.pval(p_q, digits=3), "\n")
cat("I² :", round(I2, 1), "%\n\n")

cat("Interprétation :\n")
if(p_q < 0.05) {
    cat("→ Hétérogénéité significative détectée (p < 0.05)\n")
    cat("→ Les ratios de Wald varient plus que ce qu'on attendrait par hasard\n")
    cat("→ Suggère de la pleïotropie ou des mécanismes biologiques différents\n")
} else {
    cat("→ Pas d'hétérogénéité significative (p ≥ 0.05)\n")
    cat("→ Les ratios de Wald sont homogènes\n")
}

if(I2 > 75) {
    cat("→ I² > 75% : hétérogénéité très élevée\n")
} else if(I2 > 50) {
    cat("→ I² > 50% : hétérogénéité modérée à élevée\n")
} else if(I2 > 25) {
    cat("→ I² > 25% : hétérogénéité faible à modérée\n")
} else {
    cat("→ I² < 25% : hétérogénéité faible\n")
}


#### 2. MR-Egger

La méthode **MR-Egger** est une régression **avec intercept** (contrairement à IVW qui force l'intercept à zéro). 

**Modèle** :

$$
\beta_{CAD,j} = \alpha + \theta \times \beta_{LDL,j} + \epsilon_j
$$

où :
- $\alpha$ = **intercept** (teste la pleïotropie directionnelle)
- $\theta$ = **pente** (effet causal)
- Poids : $w_j = 1/SE_{CAD,j}^2$

**Hypothèse InSIDE** : La pleïotropie horizontale (α) est **indépendante** de l'effet sur l'exposition (β_LDL).

**Interprétation** :
- Si **α ≠ 0** (p < 0.05) → Évidence de pleïotropie directionnelle moyenne
- La **pente θ** donne l'effet causal corrigé pour cette pleïotropie
- **Limite** : Moins précis que IVW (IC plus larges), suppose InSIDE



In [ ]:
%%R
## cell 19

# Régression MR-Egger = régression AVEC intercept
# Poids : 1/SE_CAD²
egger_model <- lm(BETA_CAD ~ BETA_LDL, 
                  weights = 1/SE_CAD^2, 
                  data = mr_data)

# Résultats
egger_intercept <- coef(egger_model)["(Intercept)"]
egger_slope <- coef(egger_model)["BETA_LDL"]
egger_se <- summary(egger_model)$coefficients["BETA_LDL", "Std. Error"]
egger_p <- summary(egger_model)$coefficients["BETA_LDL", "Pr(>|t|)"]
egger_intercept_p <- summary(egger_model)$coefficients["(Intercept)", "Pr(>|t|)"]

cat("Pente (effet causal) :", round(egger_slope, 4), 
    "(p", format.pval(egger_p, digits=2), ")\n")
cat("Intercept (pleïotropie) :", round(egger_intercept, 4), 
    "(p", format.pval(egger_intercept_p, digits=2), ")\n\n")

cat("Interprétation :\n")
cat("→ Pente MR-Egger :", round(egger_slope, 3), 
    "(OR =", round(exp(egger_slope), 2), ")\n")

if(egger_intercept_p < 0.05) {
    cat("→ Intercept significatif (p < 0.05) : évidence de pleïotropie directionnelle\n")
    cat("→ Les SNPs ont en moyenne un effet direct sur CAD indépendant du LDL\n")
} else {
    cat("→ Intercept non significatif : pas d'évidence de pleïotropie directionnelle\n")
}

if(abs(egger_slope - theta_ivw_val) > 0.1) {
    cat("→ Pente MR-Egger diffère de IVW : la pleïotropie biaise potentiellement IVW\n")
} else {
    cat("→ Pente MR-Egger proche de IVW : résultats cohérents\n")
}


#### 3. Weighted Median

Le **Weighted Median** estime l'effet causal comme la **médiane pondérée** des ratios de Wald. 

**Principe** :

1. Ordonner les ratios : $\theta_{(1)} \leq \theta_{(2)} \leq ... \leq \theta_{(k)}$
2. Calculer les poids cumulés normalisés : $W_j = \frac{\sum_{i=1}^{j} w_i}{\sum_{i=1}^{k} w_i}$
3. Trouver la médiane pondérée : $\theta_{WM} = \theta_{(j)}$ où $W_j \geq 0.5$ pour la première fois

**Formule** :

$$
\theta_{WM} = \text{médiane pondérée}\{\theta_j, w_j\}
$$

avec $w_j = 1/SE(\theta_j)^2$

**Robustesse** : Valide même si jusqu'à 50% des instruments sont invalides (ceux avec les plus petits poids comptent moins).

**Avantage** : Moins sensible aux outliers (SNPs avec des ratios extrêmes dus à la pleïotropie).



In [ ]:
%%R
## cell 20

# Ordonner par ratio de Wald
ord <- order(mr_data$ratio_wald)
ratios_sorted <- mr_data$ratio_wald[ord]
weights_sorted <- weights[ord]

# Poids cumulés normalisés
weights_cumsum <- cumsum(weights_sorted) / sum(weights_sorted)

# Trouver la médiane pondérée (poids cumulé = 0.5)
median_idx <- which(weights_cumsum >= 0.5)[1]
theta_wm <- ratios_sorted[median_idx]

# Bootstrap SE (approximation simple : IQR pondéré / 1.35)
q25_idx <- which(weights_cumsum >= 0.25)[1]
q75_idx <- which(weights_cumsum >= 0.75)[1]
iqr <- ratios_sorted[q75_idx] - ratios_sorted[q25_idx]
se_wm <- iqr / 1.35

# IC et p-value
wm_lower <- theta_wm - 1.96 * se_wm
wm_upper <- theta_wm + 1.96 * se_wm
z_wm <- theta_wm / se_wm
p_wm <- 2 * (1 - pnorm(abs(z_wm)))

cat("Estimé Weighted Median :", round(theta_wm, 4), "\n")
cat("SE (approximatif) :", round(se_wm, 4), "\n")
cat("IC 95% : [", round(wm_lower, 3), ";", round(wm_upper, 3), "]\n")
cat("P-value :", format.pval(p_wm, digits=2), "\n\n")

cat("Interprétation :\n")
cat("→ Weighted Median :", round(theta_wm, 3), 
    "(OR =", round(exp(theta_wm), 2), ")\n")
cat("→ Robuste même si jusqu'à 50% des instruments sont invalides\n")

if(abs(theta_wm - theta_ivw_val) < 0.1) {
    cat("→ Proche de IVW : résultats robustes\n")
} else {
    cat("→ Diffère de IVW : certains instruments influents peuvent être invalides\n")
}

### D. Synthèse et visualisation comparative

Comparons maintenant les trois estimateurs pour évaluer la **robustesse** de notre conclusion.

---

In [ ]:
%%R
## cell 21

cat("===================================================\n")
cat("4. SYNTHÈSE DES ESTIMATEURS\n")
cat("===================================================\n\n")

results_summary <- data.frame(
    Méthode = c("IVW", "MR-Egger", "Weighted Median"),
    Estimé = c(theta_ivw_val, egger_slope, theta_wm),
    OR = c(exp(theta_ivw_val), exp(egger_slope), exp(theta_wm)),
    P_value = c(p_value, egger_p, p_wm),
    Robustesse = c("Aucune pleïotropie", "Pleïotropie directionnelle", ">50% invalides")
)

print(results_summary)

# Visualisation comparative
plot_data <- data.frame(
    method = c("IVW", "MR-Egger", "Weighted Median"),
    estimate = c(theta_ivw_val, egger_slope, theta_wm),
    lower = c(theta_ivw_val - 1.96*se_ivw, 
              egger_slope - 1.96*egger_se, 
              wm_lower),
    upper = c(theta_ivw_val + 1.96*se_ivw, 
              egger_slope + 1.96*egger_se, 
              wm_upper)
)

p <- ggplot(plot_data, aes(x=method, y=estimate, color=method)) +
    geom_hline(yintercept=0, linetype="dashed", color="gray50") +
    geom_pointrange(aes(ymin=lower, ymax=upper), size=1, linewidth=1.5) +
    coord_flip() +
    labs(title="Comparaison des estimateurs MR",
         subtitle="Effet causal LDL → CAD",
         x="Méthode",
         y="Effet causal (log OR)") +
    scale_color_manual(values=c("IVW"="#2166ac", 
                                 "MR-Egger"="#b2182b", 
                                 "Weighted Median"="#35978f")) +
    theme_minimal(base_size=13) +
    theme(legend.position="none")

print(p)

cat("\n===================================================\n")
cat("CONCLUSION GÉNÉRALE\n")
cat("===================================================\n\n")

cat("→ IVW, MR-Egger et Weighted Median convergent vers un effet positif\n")
cat("→ Le LDL cholestérol a un effet causal ROBUSTE sur la CAD\n")
cat("→ L'hétérogénéité détectée (test Q) est prise en compte par les méthodes robustes\n")

if(egger_intercept_p < 0.05) {
    cat("→ Pleïotropie directionnelle détectée, mais l'effet causal reste significatif\n")
}

cat("\n→ Recommandation : L'effet causal LDL → CAD est bien établi\n")
cat("→ Cohérent avec les essais cliniques sur les statines\n")

---

### Que retenir de ces analyses ?

<div style="background-color:#fff3cd; padding:12px; border-radius:4px; border-left:4px solid #ffc107;">

**1. Test Q (Hétérogénéité)**

- **Si p < 0.05** : Les ratios de Wald varient significativement → Pleïotropie ou mécanismes différents
- **I² élevé** : Grande partie de la variabilité due à l'hétérogénéité réelle (pas au hasard)
- **Conséquence** : Justifie l'utilisation de méthodes robustes (MR-Egger, Weighted Median)

---

**2. MR-Egger**

- **Intercept ≠ 0** : Évidence de pleïotropie directionnelle (effet moyen des SNPs sur CAD indépendant du LDL)
- **Pente** : Estimation de l'effet causal corrigée pour cette pleïotropie
- **Limite** : Moins précis que IVW (IC plus larges), suppose que la pleïotropie n'est pas corrélée avec l'effet sur LDL

---

**3. Weighted Median**

- **Robuste** même si jusqu'à 50% des instruments sont invalides
- **Interprétation** : L'estimé représente la "médiane" des effets causaux pondérés
- **Si proche de IVW** : Résultats robustes à la présence d'instruments invalides

---

**4. Convergence des méthodes**

Si **IVW ≈ MR-Egger ≈ Weighted Median** :
- ✅ Forte évidence d'un effet causal robuste
- ✅ Les violations des hypothèses MR (s'il y en a) ne changent pas la conclusion

Si **les estimés divergent fortement** :
- ⚠️ Prudence : la pleïotropie peut biaiser certains estimateurs
- ⚠️ Privilégier les méthodes robustes (MR-Egger, Weighted Median)
- ⚠️ Analyses complémentaires nécessaires (ex: MR-PRESSO pour détecter outliers)

</div>

---

**Packages R spécialisés en MR** :
- [TwoSampleMR](https://mrcieu.github.io/TwoSampleMR/) — Pipeline complet avec visualisations
- [MendelianRandomization](https://cran.r-project.org/package=MendelianRandomization) — Implémentation de toutes les méthodes
- [MR-PRESSO](https://github.com/rondolab/MR-PRESSO) — Détection et correction des outliers
- [CAUSE](https://github.com/jean997/cause) — Test de causalité vs pleïotropie partagée

---

Les informations de configuration de ce notebook et de l'environnement jupyterlab associé sont disponibles à ce lien: https://github.com/CVandiedonck/MR-eQTL-env

*[last version: 05/03/2026 by Claire Vandiedonck]*

---